# Rhythm and Pause Features (IEMOCAP)

This notebook extracts pause ratios and onset-rate statistics.
Each utterance becomes one training row for downstream SER models.

In [6]:
from pathlib import Path
import os
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

import librosa
import numpy as np
import pandas as pd


In [2]:
# Configuration
REPO_ROOT = Path.cwd().parents[1]  # repo root (notebook is under feature_extraction/)
CSV_PATH = REPO_ROOT / "datasets" / "IEMOCAP" / "iemocap_full_dataset.csv"
AUDIO_ROOT = REPO_ROOT / "datasets" / "IEMOCAP"
OUT_DIR = REPO_ROOT / "extracted_features" / "rhythm_pauses"
OUT_FILE = "rhythm_pauses_features.csv"

# Audio + feature params
TARGET_SR = 16_000
FRAME_LENGTH = 2048
HOP_LENGTH = 512
TOP_DB = 40.0

EXCLUDED_EMOTIONS = {"sur", "fea", "oth", "dis"}

OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / OUT_FILE
OUT_PATH


PosixPath('/home/marcello/Speech-Emotion-Recognition/extracted_features/rhythm_pauses/rhythm_pauses_features.csv')

In [3]:
def load_audio(path: Path) -> tuple[np.ndarray, int]:
    # Load audio and resample to TARGET_SR so features are comparable
    audio, sr = librosa.load(path, sr=TARGET_SR, mono=True)
    return audio, sr


def _pause_stats(pause_durations: np.ndarray) -> dict[str, float]:
    # Summarize pause durations with NaN-safe stats
    if pause_durations.size == 0:
        return {
            "rhythm_pause_count": 0.0,
            "rhythm_pause_mean_s": float("nan"),
            "rhythm_pause_std_s": float("nan"),
            "rhythm_pause_median_s": float("nan"),
            "rhythm_pause_max_s": float("nan"),
        }

    return {
        "rhythm_pause_count": float(pause_durations.size),
        "rhythm_pause_mean_s": float(np.mean(pause_durations)),
        "rhythm_pause_std_s": float(np.std(pause_durations)),
        "rhythm_pause_median_s": float(np.median(pause_durations)),
        "rhythm_pause_max_s": float(np.max(pause_durations)),
    }


def extract_rhythm_features(audio: np.ndarray, sr: int) -> dict[str, float]:
    # Extract rhythm and pause statistics from audio
    duration = float(audio.shape[0]) / float(sr) if sr > 0 else 0.0
    if duration <= 0.0:
        return {
            "rhythm_pause_ratio": 0.0,
            "rhythm_speech_ratio": 0.0,
            "rhythm_pause_count": 0.0,
            "rhythm_pause_mean_s": float("nan"),
            "rhythm_pause_std_s": float("nan"),
            "rhythm_pause_median_s": float("nan"),
            "rhythm_pause_max_s": float("nan"),
            "rhythm_onset_rate_hz": 0.0,
            "rhythm_onset_count": 0.0,
        }

    intervals = librosa.effects.split(
        audio,
        top_db=TOP_DB,
        frame_length=FRAME_LENGTH,
        hop_length=HOP_LENGTH,
    )

    speech_samples = np.sum(intervals[:, 1] - intervals[:, 0]) if intervals.size else 0
    speech_duration = float(speech_samples) / float(sr)
    total_silence = max(duration - speech_duration, 0.0)
    pause_ratio = total_silence / duration if duration > 0.0 else 0.0

    if intervals.shape[0] >= 2:
        gaps = intervals[1:, 0] - intervals[:-1, 1]
        gaps = gaps[gaps > 0]
        pause_durations = gaps.astype(float) / float(sr)
    else:
        pause_durations = np.array([], dtype=float)

    onsets = librosa.onset.onset_detect(
        y=audio,
        sr=sr,
        hop_length=HOP_LENGTH,
    )
    onset_count = float(onsets.size)
    onset_rate = onset_count / duration if duration > 0.0 else 0.0

    features = {
        "rhythm_pause_ratio": float(pause_ratio),
        "rhythm_speech_ratio": float(1.0 - pause_ratio),
        "rhythm_onset_rate_hz": float(onset_rate),
        "rhythm_onset_count": float(onset_count),
    }
    features.update(_pause_stats(pause_durations))
    return features


In [4]:
df = pd.read_csv(CSV_PATH)  # metadata for paths + labels
df["emotion"] = df["emotion"].astype(str).str.strip().str.lower()

# Filter: keep xxx, exclude selected classes, and enforce agreement for labeled classes.
# This keeps unlabeled (xxx) examples while dropping sur/fea/oth/dis.
df = df[~df["emotion"].isin(EXCLUDED_EMOTIONS)].copy()
df = df[(df["emotion"] == "xxx") | (df["agreement"] > 0)].copy()
df.shape


(7532, 7)

In [5]:
CPU_COUNT = os.cpu_count() or 1
COMPUTE_DEVICE = "cpu"  # Force CPU for this CPU-bound extractor
NUM_WORKERS = max(1, CPU_COUNT - 2)
PROGRESS_MIN_INTERVAL = 1.0

print(f"Compute device: {COMPUTE_DEVICE} | extractor_backend=cpu | workers={NUM_WORKERS}")


def process_row(row: dict[str, object]) -> tuple[dict[str, float | str | int] | None, str | None]:
    rel_path = str(row["path"])
    audio_path = AUDIO_ROOT / rel_path
    if not audio_path.exists():
        return None, str(audio_path)

    audio, sr = load_audio(audio_path)
    duration_s = audio.shape[0] / sr
    features = extract_rhythm_features(audio, sr)

    record: dict[str, float | str | int] = {
        "path": rel_path,
        "session": int(row["session"]),
        "method": str(row["method"]),
        "gender": str(row["gender"]),
        "emotion": str(row["emotion"]),
        "n_annotators": int(row["n_annotators"]),
        "agreement": int(row["agreement"]),
        "duration_s": float(duration_s),
    }
    record.update(features)
    return record, None


rows: list[dict[str, float | str | int]] = []
missing: list[str] = []
records = df.to_dict(orient="records")

if NUM_WORKERS > 1:
    with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
        mapped = executor.map(process_row, records)
        for record, missing_path in tqdm(mapped, total=len(records), desc="Extracting", unit="file", mininterval=PROGRESS_MIN_INTERVAL):
            if missing_path is not None:
                missing.append(missing_path)
                continue
            if record is not None:
                rows.append(record)
else:
    for record in tqdm(records, total=len(records), desc="Extracting", unit="file", mininterval=PROGRESS_MIN_INTERVAL):
        row_result, missing_path = process_row(record)
        if missing_path is not None:
            missing.append(missing_path)
            continue
        if row_result is not None:
            rows.append(row_result)

feature_df = pd.DataFrame(rows)
feature_df.to_csv(OUT_PATH, index=False)

print(f"Saved: {OUT_PATH}")
print(f"Workers used: {NUM_WORKERS} (cpu_count={CPU_COUNT})")
if missing:
    print(f"Missing audio files: {len(missing)}")
feature_df.shape


Saved: /home/marcello/Speech-Emotion-Recognition/extracted_features/rhythm_pauses/rhythm_pauses_features.csv
Workers used: 62 (cpu_count=64)


(7532, 17)